<div dir="rtl" align="right">

# ترميزُ One-Hot لِوسومِ EEG

**مجموعةُ البياناتِ**: MOABB BNCI2014-001 (تخيّلٌ حركيّ)  
**القنواتُ**: 22 قناةً  
**معدّلُ أخذِ العيناتِ**: 250 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نُوضّحُ ترميزَ one-hot لِوسومِ EEG مُتعدّدةِ الفئاتِ بِاستخدامِ جميعِ الفئاتِ الأربعِ للتخيّلِ الحركيّ.

## ماذا يَعمَلُ هذا الدفترُ؟

يُطبّقُ LabelEncoder ثمَّ OneHotEncoder لِتحويلِ الوسومِ النصّيّةِ إلى تمثيلٍ بِمصفوفةٍ ثنائيّةٍ.

## المُخرجاتُ المُتوقّعةُ

- مخططٌ شريطيٌّ يُظهرُ توزيعَ جميعِ الفئاتِ الأربعِ
- خريطةٌ حراريّةٌ لِمصفوفةِ الترميزِ one-hot لِأوّلِ 20 تجربةً
- كلُّ تجربةٍ لها '1' واحدةٌ بالضبطِ في عمودِ فئتِها

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ |
| --- | --- |
| fmin | 8 |
| fmax | 32 |
| n_classes | 4 |
| encoder | LabelEncoder, OneHotEncoder |

</div>


<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>


In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


<div dir="rtl" align="right">

## 2. تحميلُ مجموعةِ بياناتِ MOABB

تُنزّلُ MOABB البياناتِ تلقائيّاً عندَ أوّلِ استدعاءٍ (حوالي 44 ميجابايت).

</div>


In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=4, fmin=8, fmax=32)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


In [ ]:
# Using all 4 classes (no filtering)
print(f'Classes: {np.unique(labels)}')
print(f'Class distribution: {[(c, np.sum(labels == c)) for c in np.unique(labels)]}')


<div dir="rtl" align="right">

## 3. استكشافُ البياناتِ

</div>


In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


<div dir="rtl" align="right">

## 4. تطبيقُ ترميزِ one-hot

</div>


In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)

onehot_encoder = OneHotEncoder(sparse_output=False)
onehot_matrix = onehot_encoder.fit_transform(labels_encoded.reshape(-1, 1))

classes = label_encoder.classes_
print(f'Encoded labels: {np.unique(labels_encoded)}')
print(f'One-hot matrix shape: {onehot_matrix.shape}')
print(f'Classes: {classes}')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- جميعُ الفئاتِ الأربعِ ينبغي أن تَكونَ لها أعدادٌ مُتساويةٌ تقريباً (مجموعةُ بياناتٍ مُتوازنةٌ)
- مصفوفةُ one-hot تُظهرُ '1' واحدةً لِكلِّ صفٍّ، مُؤكّدةً الترميزَ الصحيحَ
- كلُّ عمودٍ يُقابلُ فئةً واحدةً

</div>


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

counts = [np.sum(labels == c) for c in classes]

fig = make_subplots(rows=2, cols=1, subplot_titles=(
    'Class Distribution (4 classes)',
    'One-Hot Encoded Matrix (first 20 trials)'))

fig.add_trace(go.Bar(x=list(classes), y=counts, marker_color='steelblue', name='Count'), row=1, col=1)
fig.add_trace(go.Heatmap(z=onehot_matrix[:20, :], x=list(classes),
    colorscale='Blues', name='One-Hot', showscale=True), row=2, col=1)

fig.update_xaxes(title_text='Class', row=1, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_xaxes(title_text='Class', row=2, col=1)
fig.update_yaxes(title_text='Trial', row=2, col=1)
fig.update_layout(height=800, showlegend=False, title_text='One-Hot Encoding of EEG Labels')
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- ترميزُ one-hot يُحوّلُ الوسومَ الفئويّةَ إلى مصفوفةٍ ثنائيّةٍ مُناسبةٍ لِلشبكاتِ العصبيّةِ
- كلُّ صفٍّ له '1' واحدةٌ بالضبطِ، تُمثّلُ الفئةَ النشطةَ
- LabelEncoder يُرسِمُ السلاسلَ إلى أعدادٍ صحيحةٍ، و OneHotEncoder يُوسّعُ الأعدادَ إلى متجهاتٍ ثنائيّةٍ

</div>
